# Design Demo Video Frame

Use this notebook to tune the demo-video layout before rendering a full MP4. The renderer is session-time native: choose one `preview_session_time_s`, render all components at that session time, and adjust the demo-video YAML until the layout looks right.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt

from multimodal_sync.config import config_root, load_config
from multimodal_sync.demo_video.context import DemoSessionContext
from multimodal_sync.demo_video.renderer import DemoVideoRenderer
from multimodal_sync.demo_video.timeline import SessionRenderTimeline

## Paths

In [ ]:
session_basepath = Path(os.environ.get("MULTIMODAL_SYNC_EXAMPLE_SESSION", "/path/to/example_session_01"))

if (Path.cwd() / "sync_analysis" / "configs").is_dir():
    sync_analysis_root = Path.cwd() / "sync_analysis"
elif (Path.cwd().parent / "configs").is_dir():
    sync_analysis_root = Path.cwd().parent
else:
    raise FileNotFoundError("Could not locate sync_analysis/configs from the current working directory")

session_config_path = sync_analysis_root / "configs" / "example_session_01_50hz.yaml"
demo_video_config_path = sync_analysis_root / "configs" / "demo_video" / "example_video_audio_waveform.yaml"

## Build Renderer

In [ ]:
session_config = load_config(session_config_path)
demo_video_config = config_root(load_config(demo_video_config_path))

clip_start_session_s = 0.0
clip_end_session_s = 15.0
preview_session_time_s = 8.0

context = DemoSessionContext(session_basepath=session_basepath, session_config=session_config)
timeline = SessionRenderTimeline(
    clip_start_session_s=clip_start_session_s,
    clip_end_session_s=clip_end_session_s,
    output_fps=float(demo_video_config["video"].get("output_fps", 30)),
    playback_speed=float(demo_video_config["video"].get("playback_speed", 1.0)),
)
renderer = DemoVideoRenderer(
    context=context,
    demo_video_config=demo_video_config,
    timeline=timeline,
)
timeline.to_dict()

## Preview One Frame

In [ ]:
preview_rgb = renderer.preview_frame(
    session_time_s=preview_session_time_s,
    show_component_bounds=True,
)

fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(preview_rgb)
ax.set_axis_off()
plt.show()